# Phase 2C.1 Smoke Acceptance

This notebook verifies the first controlled Phase 2C smoke batch after its artifacts exist. It is not final Phase 2C signoff.

The smoke goal is infrastructure and orchestration validation: artifact shape, lifecycle accounting, theme rotation, cost tracking, and failure visibility. It does not claim alpha discovery or production research quality.

## Configuration

Set `SMOKE_ARTIFACT_PATH` to the smoke artifact directory before running after a batch. If it is left blank, the notebook will try a conservative discovery pass under `raw_payloads/` and otherwise return `PARTIAL / NOT_RUN` rather than failing.

In [13]:
from __future__ import annotations

import ast
import json
import re
import subprocess
from collections import Counter
from pathlib import Path
from pprint import pprint

try:
    import pandas as pd
except ImportError:  # Notebook remains runnable without pandas.
    pd = None

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / '.git').exists():
    REPO_ROOT = REPO_ROOT.parent

SMOKE_ARTIFACT_PATH = ''  # e.g. 'raw_payloads/batch_<phase2c_smoke_batch_id>'
SMOKE_COST_CAP_USD = 1.00
KNOWN_UNRELATED_DIRTY_PREFIXES = ('.DS_Store', '.claude/')
EXPECTED_STAGE_LABEL_PATTERNS = ('PHASE2C', 'STAGE2C', 'D6_STAGE2C', 'phase2c', 'stage2c')

GATES = {f'G{i}': {'passed': False, 'failed': False, 'status': 'PENDING', 'detail': ''} for i in range(1, 11)}
warnings: list[str] = []
failures: list[str] = []

def rel(path: Path | str) -> str:
    p = Path(path)
    try:
        return str(p.resolve().relative_to(REPO_ROOT.resolve()))
    except Exception:
        return str(p)

def run_git(args: list[str]) -> str:
    try:
        out = subprocess.run(['git', *args], cwd=REPO_ROOT, text=True, capture_output=True, check=False, timeout=10)
        return (out.stdout or out.stderr).strip()
    except Exception as exc:
        warnings.append(f'git {args} failed: {exc}')
        return ''

def mark_gate(gate: str, status: str, detail: str = '') -> None:
    GATES[gate]['status'] = status
    GATES[gate]['passed'] = status in {'PASS', 'WARN'}
    GATES[gate]['failed'] = status == 'FAIL'
    GATES[gate]['detail'] = detail
    if status == 'FAIL':
        failures.append(f'{gate}: {detail}')
    elif status == 'WARN':
        warnings.append(f'{gate}: {detail}')

def display_rows(rows: list[dict]) -> None:
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        pprint(rows)

print(f'Repo root: {REPO_ROOT}')
print(f'Smoke cost cap: ${SMOKE_COST_CAP_USD:.2f}')

Repo root: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline
Smoke cost cap: $1.00


## Environment and Git State

Capture branch, HEAD, and dirty files. Known unrelated local files are surfaced as warnings, not automatic failures.

In [14]:
branch = run_git(['branch', '--show-current']) or 'unknown'
head_commit = run_git(['rev-parse', '--short', 'HEAD']) or 'unknown'
status_porcelain = run_git(['status', '--short'])
status_lines = [line for line in status_porcelain.splitlines() if line.strip()]

known_dirty = []
other_dirty = []
for line in status_lines:
    path = line[3:] if len(line) > 3 else line
    if any(path == prefix.rstrip('/') or path.startswith(prefix) for prefix in KNOWN_UNRELATED_DIRTY_PREFIXES):
        known_dirty.append(line)
    else:
        other_dirty.append(line)

git_state = {
    'branch': branch,
    'head_commit': head_commit,
    'dirty_file_count': len(status_lines),
    'known_unrelated_dirty': known_dirty,
    'other_dirty': other_dirty,
}
pprint(git_state)

if status_lines:
    warnings.append(f'Working tree has {len(status_lines)} dirty entries; review before treating this as immutable acceptance evidence.')
mark_gate('G1', 'PASS', 'Git environment captured')

{'branch': 'claude/setup-structure-validators-JNqoI',
 'dirty_file_count': 5,
 'head_commit': '613da3f',
 'known_unrelated_dirty': ['?? .claude/'],
 'other_dirty': ['M .DS_Store',
                 '?? docs/d7_stage2c/D7_STAGE2C_PATCH_REPORT.md',
                 '?? docs/d7_stage2c/stage2c_candidate_worksheet.md',
                 '?? docs/test_notebooks/PHASE2C_1_smoke_acceptance.ipynb']}


## Static Guardrail Checks

These checks inspect code and docs only. They do not execute batch jobs, API calls, or data downloads.

In [15]:
def literal_assignments(path: Path) -> dict[str, object]:
    tree = ast.parse(path.read_text(encoding='utf-8'))
    values = {}
    for node in tree.body:
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    try:
                        values[target.id] = ast.literal_eval(node.value)
                    except Exception:
                        pass
        elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name):
            try:
                values[node.target.id] = ast.literal_eval(node.value)
            except Exception:
                pass
    return values

stage2c_vals = literal_assignments(REPO_ROOT / 'agents/proposer/stage2c_batch.py')
stage2d_vals = literal_assignments(REPO_ROOT / 'agents/proposer/stage2d_batch.py')
theme_vals = literal_assignments(REPO_ROOT / 'agents/themes.py')
themes = tuple(theme_vals.get('THEMES', ()))

def _is_acceptance_or_signoff_doc(p: Path) -> bool:
    name = p.name
    return ('PHASE2C' in name and 'SIGNOFF' in name) or 'smoke_acceptance' in name

py_md_files = [
    p for p in REPO_ROOT.rglob('*')
    if p.suffix in {'.py', '.md'}
    and '.git' not in p.parts
    and not _is_acceptance_or_signoff_doc(p)
]
phase2_blueprint_refs = []
for p in py_md_files:
    try:
        text = p.read_text(encoding='utf-8')
    except UnicodeDecodeError:
        continue
    if 'PHASE2_BLUEPRINT_v2' in text:
        phase2_blueprint_refs.append(rel(p))

bulk_text = (REPO_ROOT / 'ingestion/bulk_download.py').read_text(encoding='utf-8')
bulk_checks = {
    '--force option': "'--force'" in bulk_text or '"--force"' in bulk_text,
    'archive before overwrite': all(token in bulk_text for token in ('ARCHIVE_DIR', 'copy2', 'args.force')),
    'staging path': 'staging_path' in bulk_text,
    'atomic promote replace': '.replace(output_path)' in bulk_text,
}

static_rows = [
    {'check': 'stage2c THEME_CYCLE_LEN == 5', 'pass': stage2c_vals.get('THEME_CYCLE_LEN') == 5, 'value': stage2c_vals.get('THEME_CYCLE_LEN')},
    {'check': 'stage2d THEME_CYCLE_LEN == 5', 'pass': stage2d_vals.get('THEME_CYCLE_LEN') == 5, 'value': stage2d_vals.get('THEME_CYCLE_LEN')},
    {'check': 'len(THEMES) == 6', 'pass': len(themes) == 6, 'value': len(themes)},
    {'check': 'multi_factor_combination present', 'pass': 'multi_factor_combination' in themes, 'value': themes},
    {'check': 'multi_factor_combination excluded by cycle length', 'pass': len(themes) > 5 and themes[5] == 'multi_factor_combination', 'value': themes[:5]},
    {'check': 'no PHASE2_BLUEPRINT_v2 refs in .py/.md', 'pass': len(phase2_blueprint_refs) == 0, 'value': phase2_blueprint_refs},
    *({'check': f'bulk_download {name}', 'pass': ok, 'value': ok} for name, ok in bulk_checks.items()),
]
display_rows(static_rows)

if all(row['pass'] for row in static_rows):
    mark_gate('G2', 'PASS', 'Static guardrails pass')
else:
    mark_gate('G2', 'FAIL', 'One or more static guardrails failed')

,check,pass,value
0,stage2c THEME_CYCLE_LEN == 5,True,5
1,stage2d THEME_CYCLE_LEN == 5,True,5
2,len(THEMES) == 6,True,6
3,multi_factor_combination present,True,"(momentum, mean_reversion, volatility_regime, ..."
4,multi_factor_combination excluded by cycle length,True,"(momentum, mean_reversion, volatility_regime, ..."
5,no PHASE2_BLUEPRINT_v2 refs in .py/.md,True,[]
6,bulk_download --force option,True,True
7,bulk_download archive before overwrite,True,True
8,bulk_download staging path,True,True
9,bulk_download atomic promote replace,True,True


## Smoke Batch Artifact Discovery

The notebook inspects an existing artifact directory. If no smoke artifact is found, the run is `PARTIAL / NOT_RUN`, not `FAIL`.

In [16]:
def candidate_artifact_dirs() -> list[Path]:
    candidates = []
    raw = REPO_ROOT / 'raw_payloads'
    if raw.exists():
        for p in raw.iterdir():
            name = p.name.lower()
            looks_like_smoke = 'smoke' in name or 'phase2c' in name or 'stage2c' in name or name.startswith('batch_')
            has_stage2c_summary = any('stage2c' in s.name.lower() and 'summary' in s.name.lower() for s in p.rglob('*.json'))
            if p.is_dir() and looks_like_smoke and has_stage2c_summary:
                candidates.append(p)
    return sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)

artifact_path = None
if SMOKE_ARTIFACT_PATH.strip():
    candidate = (REPO_ROOT / SMOKE_ARTIFACT_PATH).resolve()
    if candidate.exists() and candidate.is_dir():
        artifact_path = candidate
    else:
        warnings.append(f'Configured smoke artifact path does not exist: {SMOKE_ARTIFACT_PATH}')
else:
    discovered = candidate_artifact_dirs()
    if discovered:
        artifact_path = discovered[0]
        warnings.append(f'No SMOKE_ARTIFACT_PATH configured; using most recent candidate: {rel(artifact_path)}')

inspected_artifact_path = rel(artifact_path) if artifact_path else None
print(f'Inspected artifact path: {inspected_artifact_path or "NOT_RUN"}')

if artifact_path is None:
    mark_gate('G3', 'WARN', 'No smoke artifact path configured or discovered')
else:
    mark_gate('G3', 'PASS', f'Artifact path discovered: {inspected_artifact_path}')

Inspected artifact path: NOT_RUN


## Artifact Completeness Checks

Check for summaries, proposer prompt/response files, optional critic artifacts, and aggregate result JSON where present.

In [17]:
def load_json_file(path: Path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        warnings.append(f'Could not load JSON {rel(path)}: {exc}')
        return None

summary_files = []
aggregate_files = []
prompt_files = []
response_files = []
critic_prompt_files = []
critic_response_files = []
critic_result_files = []
summary = None

if artifact_path is not None:
    json_files = list(artifact_path.rglob('*.json'))
    summary_files = sorted([p for p in json_files if 'summary' in p.name.lower()])
    aggregate_files = sorted([p for p in json_files if any(tok in p.name.lower() for tok in ('aggregate', 'result', 'record')) and p not in summary_files])
    prompt_files = sorted([p for p in artifact_path.rglob('*prompt*.txt') if 'critic' not in str(p).lower()])
    response_files = sorted([p for p in artifact_path.rglob('*response*') if p.is_file() and 'critic' not in str(p).lower()])
    critic_prompt_files = sorted([p for p in artifact_path.rglob('*critic*prompt*') if p.is_file()])
    critic_response_files = sorted([p for p in artifact_path.rglob('*critic*response*') if p.is_file()])
    critic_result_files = sorted([p for p in artifact_path.rglob('*critic*result*') if p.is_file()])
    if summary_files:
        summary = load_json_file(summary_files[0])
    elif aggregate_files:
        summary = load_json_file(aggregate_files[0])

critic_stage_detected = bool(critic_prompt_files or critic_response_files or critic_result_files)
artifact_rows = [
    {'artifact': 'summary JSON', 'count': len(summary_files), 'status': 'present' if summary_files else 'missing'},
    {'artifact': 'proposer prompts', 'count': len(prompt_files), 'status': 'present' if prompt_files else 'missing'},
    {'artifact': 'proposer responses', 'count': len(response_files), 'status': 'present' if response_files else 'missing'},
    {'artifact': 'critic prompts', 'count': len(critic_prompt_files), 'status': 'present' if critic_prompt_files else ('skipped' if not critic_stage_detected else 'missing')},
    {'artifact': 'critic responses/results', 'count': len(critic_response_files) + len(critic_result_files), 'status': 'present' if (critic_response_files or critic_result_files) else ('skipped' if not critic_stage_detected else 'missing')},
    {'artifact': 'aggregate/result/record JSON', 'count': len(aggregate_files), 'status': 'present' if aggregate_files else 'not generated or missing'},
]
display_rows(artifact_rows)

if artifact_path is None:
    mark_gate('G8', 'WARN', 'Artifact completeness not run because no artifact path was available')
elif summary is None:
    mark_gate('G8', 'FAIL', 'No loadable summary or record JSON found')
elif not prompt_files or not response_files:
    mark_gate('G8', 'WARN', 'Summary exists but prompt/response files are incomplete for smoke scope')
else:
    mark_gate('G8', 'PASS', 'Artifacts complete enough for configured smoke scope')

,artifact,count,status
0,summary JSON,0,missing
1,proposer prompts,0,missing
2,proposer responses,0,missing
3,critic prompts,0,skipped
4,critic responses/results,0,skipped
5,aggregate/result/record JSON,0,not generated or missing


## Batch Execution Summary

Load available summary fields defensively. Missing fields become warnings rather than crashes.

In [18]:
def first_present(mapping: dict, keys: tuple[str, ...], default=None):
    for key in keys:
        if isinstance(mapping, dict) and key in mapping:
            return mapping[key]
    return default

def find_records(obj):
    if isinstance(obj, dict):
        for key in ('call_summaries', 'calls', 'records', 'call_records', 'attempts', 'results', 'candidates'):
            val = obj.get(key)
            if isinstance(val, list) and all(isinstance(x, dict) for x in val):
                return val
        for val in obj.values():
            found = find_records(val)
            if found:
                return found
    return []

records = find_records(summary) if isinstance(summary, dict) else []
attempted = first_present(summary or {}, ('hypotheses_attempted', 'attempted', 'attempted_count', 'completed_call_count', 'issued_call_count', 'batch_size'), len(records) or None)
parsed = first_present(summary or {}, ('parsed', 'parsed_count', 'parse_success_count'), None)
valid = first_present(summary or {}, ('valid', 'valid_count', 'pending_backtest_count'), None)
errored = first_present(summary or {}, ('errored', 'error_count', 'api_error_count'), None)
retry_count = first_present(summary or {}, ('retry_count', 'total_retries'), None)

lifecycle_counter = Counter()
truncation_flags = []
for rec in records:
    # Skip truncated_by_cap entries: they have lifecycle_state=None and
    # represent unissued slots after EARLY_STOP, not real lifecycle outcomes.
    trunc = first_present(rec, ('truncated_by_cap', 'truncated', 'was_truncated'), None)
    if trunc:
        truncation_flags.append(True)
        continue
    # Use lifecycle_state directly, not first_present fallback to valid_status
    # (which would mis-count truncated_by_cap as a lifecycle state).
    state = rec.get('lifecycle_state')
    if state:
        lifecycle_counter[str(state)] += 1
        truncation_flags.append(False)

summary_report = {
    'attempted': attempted,
    'parsed': parsed,
    'valid': valid,
    'errored': errored,
    'retry_count': retry_count,
    'record_count': len(records),
    'lifecycle_counts': dict(lifecycle_counter),
    'truncation_true_count': sum(truncation_flags),
}
pprint(summary_report)

if summary is None:
    mark_gate('G4', 'WARN', 'Lifecycle reconciliation not run; no summary loaded')
elif attempted is None:
    mark_gate('G4', 'WARN', 'Attempted count missing from summary')
elif lifecycle_counter and sum(lifecycle_counter.values()) != attempted:
    mark_gate('G4', 'FAIL', f'Lifecycle counts sum to {sum(lifecycle_counter.values())}, attempted={attempted}')
elif records and attempted != len(records):
    mark_gate('G4', 'WARN', f'Record count {len(records)} differs from attempted={attempted}; lifecycle counts unavailable or partial')
else:
    mark_gate('G4', 'PASS', 'Lifecycle counts reconcile or record count matches attempted')

{'attempted': None,
 'errored': None,
 'lifecycle_counts': {},
 'parsed': None,
 'record_count': 0,
 'retry_count': None,
 'truncation_true_count': 0,
 'valid': None}


## Theme Rotation Validation

Confirm operational calls use only the first five themes and that `multi_factor_combination` receives zero calls under the current boundary.

In [19]:
operational_themes = themes[:5]
excluded_theme = 'multi_factor_combination'
theme_counter = Counter()

for rec in records:
    theme = first_present(rec, ('theme', 'theme_name', 'assigned_theme'), None)
    if theme:
        theme_counter[str(theme)] += 1

if not theme_counter and attempted:
    for idx in range(1, int(attempted) + 1):
        theme_counter[operational_themes[(idx - 1) % 5]] += 1
    warnings.append('Theme records missing; inferred expected cyclic distribution from attempted count.')

theme_rows = [{'theme': t, 'count': theme_counter.get(t, 0), 'operational': t in operational_themes} for t in themes]
display_rows(theme_rows)

used_themes = {t for t, n in theme_counter.items() if n > 0}
bad_themes = sorted(used_themes - set(operational_themes))
op_counts = [theme_counter.get(t, 0) for t in operational_themes]
balanced = True
if op_counts and sum(op_counts) > 0:
    balanced = max(op_counts) - min(op_counts) <= 1

if not theme_counter:
    mark_gate('G5', 'WARN', 'No theme data available to validate rotation')
elif excluded_theme in bad_themes or theme_counter.get(excluded_theme, 0) != 0:
    mark_gate('G5', 'FAIL', 'multi_factor_combination was used operationally')
elif bad_themes:
    mark_gate('G5', 'FAIL', f'Unexpected operational themes used: {bad_themes}')
elif not balanced:
    mark_gate('G5', 'FAIL', f'Operational theme distribution is not roughly balanced: {dict(theme_counter)}')
else:
    mark_gate('G5', 'PASS', 'Theme rotation matches 5-theme operational boundary')

,theme,count,operational
0,momentum,0,True
1,mean_reversion,0,True
2,volatility_regime,0,True
3,volume_divergence,0,True
4,calendar_effect,0,True
5,multi_factor_combination,0,False


## Cost Tracking Validation

Compare known estimated or actual cost against the editable smoke cap. Missing cost is a warning gate, not a hard failure.

In [20]:
def numeric_values_for_keys(obj, key_patterns):
    found = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            lk = key.lower()
            if any(pat in lk for pat in key_patterns) and isinstance(value, (int, float)):
                found.append((key, float(value)))
            found.extend(numeric_values_for_keys(value, key_patterns))
    elif isinstance(obj, list):
        for item in obj:
            found.extend(numeric_values_for_keys(item, key_patterns))
    return found

# Use top-level total_actual_cost_usd directly (with estimated fallback) to
# avoid pulling in derived metrics like cost_ratio (which is inf when
# total_actual_cost_usd == 0 under stub backend).
cost_values = numeric_values_for_keys(summary, ('cost_usd',)) if summary is not None else []
estimated_costs = [(k, v) for k, v in cost_values if 'estimated' in k.lower() or 'est_' in k.lower()]
actual_costs = [(k, v) for k, v in cost_values if 'actual' in k.lower()]

total_actual = summary.get('total_actual_cost_usd') if isinstance(summary, dict) else None
total_estimated = summary.get('total_estimated_cost_usd') if isinstance(summary, dict) else None
if total_actual is None and actual_costs:
    total_actual = max(v for _, v in actual_costs)
if total_estimated is None and estimated_costs:
    total_estimated = max(v for _, v in estimated_costs)
known_cost = total_actual if total_actual is not None else total_estimated

cost_report = {
    'smoke_cost_cap_usd': SMOKE_COST_CAP_USD,
    'total_estimated_cost_usd': total_estimated,
    'total_actual_cost_usd': total_actual,
    'cost_field_count': len(cost_values),
}
pprint(cost_report)

if known_cost is None:
    mark_gate('G6', 'WARN', 'Cost fields not available; cannot prove under cap')
elif known_cost <= SMOKE_COST_CAP_USD:
    mark_gate('G6', 'PASS', f'Known cost ${known_cost:.4f} <= cap ${SMOKE_COST_CAP_USD:.2f}')
else:
    mark_gate('G6', 'FAIL', f'Known cost ${known_cost:.4f} exceeds cap ${SMOKE_COST_CAP_USD:.2f}')

{'cost_field_count': 0,
 'smoke_cost_cap_usd': 1.0,
 'total_actual_cost_usd': None,
 'total_estimated_cost_usd': None}


## Parse, Validation, and Failure Review

Summarize infrastructure failure modes and surface representative examples.

In [21]:
failure_terms = ('error', 'fail', 'invalid', 'empty', 'duplicate', 'truncated')
failure_records = []
for rec in records:
    state_text = ' '.join(str(first_present(rec, keys, '')) for keys in [
        ('lifecycle_state', 'status', 'valid_status', 'candidate_status'),
        ('error_category', 'error_signature', 'exception_type'),
        ('parse_error', 'validation_error', 'api_error'),
    ]).lower()
    if any(term in state_text for term in failure_terms):
        failure_records.append(rec)

failure_rows = []
for rec in failure_records[:10]:
    failure_rows.append({
        'call': first_present(rec, ('call_index', 'attempt', 'k', 'call_id', 'id'), None),
        'theme': first_present(rec, ('theme', 'theme_name', 'assigned_theme'), None),
        'state': first_present(rec, ('lifecycle_state', 'status', 'valid_status', 'candidate_status'), None),
        'error_category': first_present(rec, ('error_category', 'exception_type'), None),
        'error_signature': first_present(rec, ('error_signature', 'parse_error', 'validation_error', 'api_error'), None),
    })

failure_summary = {
    'failure_record_count': len(failure_records),
    'representative_examples_shown': len(failure_rows),
}
pprint(failure_summary)
display_rows(failure_rows)

unexplained_infra_errors = []
for rec in failure_records:
    sig = first_present(rec, ('error_signature', 'parse_error', 'validation_error', 'api_error'), None)
    cat = first_present(rec, ('error_category', 'exception_type'), None)
    if not sig and not cat:
        unexplained_infra_errors.append(rec)

# Stub-mode detection: under --dry-run the StubProposerBackend produces
# lifecycle outcomes (invalid_dsl, duplicate, rejected_complexity) without
# error_category / error_signature fields. These are intentional stub
# outputs, not unhandled infrastructure errors. G7 should accept the
# absence of error metadata under stub mode; live Sonnet runs continue
# to require error metadata for unexplained failures.
is_stub_run = bool(summary.get('dry_run', False)) if isinstance(summary, dict) else False

if summary is None:
    mark_gate('G7', 'WARN', 'Failure review not run; no summary loaded')
elif is_stub_run:
    # Under stub mode, the gate validates absence of unhandled exceptions
    # / abort artifacts (checked separately in artifact discovery), not
    # error-metadata richness on lifecycle outcomes.
    mark_gate('G7', 'PASS', f'Stub-mode run: {len(failure_records)} lifecycle outcomes recorded; error-metadata richness check skipped (stub backend does not produce category/signature)')
elif unexplained_infra_errors:
    mark_gate('G7', 'FAIL', f'{len(unexplained_infra_errors)} failure records lack error category/signature')
else:
    mark_gate('G7', 'PASS', 'No unexplained infrastructure errors found in available records')

{'failure_record_count': 0, 'representative_examples_shown': 0}


""


## Candidate Quality Smoke Summary

This is a smoke-quality summary only. Zero strong alpha candidates can still be acceptable if infrastructure behavior is clean and the result is explainable.

In [22]:
usable_states = {'pending_backtest', 'valid', 'ok', 'accepted'}
duplicate_states = {'duplicate', 'invalid_duplicate'}
invalid_states = {'invalid_dsl', 'invalid_schema', 'rejected_complexity', 'invalid'}

quality_counter = Counter()
for rec in records:
    state = str(first_present(rec, ('lifecycle_state', 'status', 'valid_status', 'candidate_status'), '')).lower()
    if state in usable_states:
        quality_counter['usable_candidates'] += 1
    if state in duplicate_states or 'duplicate' in state:
        quality_counter['duplicate_candidates'] += 1
    if state in invalid_states or 'invalid' in state:
        quality_counter['invalid_candidates'] += 1
    if first_present(rec, ('dsl', 'strategy_dsl', 'compiled_dsl'), None):
        quality_counter['dsl_candidates_present'] += 1

pprint(dict(quality_counter))
if records and quality_counter.get('usable_candidates', 0) == 0:
    warnings.append('No usable candidates found. This does not by itself fail smoke acceptance if infrastructure gates are clean.')

{}


## Acceptance Gates

G1 git/artifact environment captured; G2 static guardrails pass; G3 smoke artifacts discoverable; G4 lifecycle counts reconcile; G5 theme rotation matches the 5-theme boundary; G6 cost under cap or unavailable with warning; G7 no unexplained infrastructure errors; G8 artifacts complete enough for smoke scope; G9 no canonical data write path touched; G10 final decision generated.

In [23]:
canonical_data_paths = [
    REPO_ROOT / 'data/raw/btcusdt_1h.parquet',
    REPO_ROOT / 'data/raw/archive',
]
canonical_touched = []
if artifact_path is not None:
    try:
        artifact_resolved = artifact_path.resolve()
        for p in canonical_data_paths:
            if p.exists():
                try:
                    artifact_resolved.relative_to(p.resolve())
                    canonical_touched.append(rel(p))
                except ValueError:
                    pass
    except Exception as exc:
        warnings.append(f'Could not compare artifact path to canonical data paths: {exc}')

if canonical_touched:
    mark_gate('G9', 'FAIL', f'Smoke artifact path is under canonical data write path: {canonical_touched}')
else:
    mark_gate('G9', 'PASS', 'No canonical data write path touched by inspected artifact path')

display_rows([{'gate': gate, **info} for gate, info in GATES.items()])

,gate,passed,failed,status,detail
0,G1,True,False,PASS,Git environment captured
1,G2,True,False,PASS,Static guardrails pass
2,G3,True,False,WARN,No smoke artifact path configured or discovered
3,G4,True,False,WARN,Lifecycle reconciliation not run; no summary l...
4,G5,True,False,WARN,No theme data available to validate rotation
5,G6,True,False,WARN,Cost fields not available; cannot prove under cap
6,G7,True,False,WARN,Failure review not run; no summary loaded
7,G8,True,False,WARN,Artifact completeness not run because no artif...
8,G9,True,False,PASS,No canonical data write path touched by inspec...
9,G10,False,False,PENDING,


## Final Verdict

`PASS` requires all required smoke gates to pass. `PARTIAL PASS` is used when infrastructure mostly works but non-critical fields or artifacts are missing or not run. `FAIL` is used for orchestration breakage, unusable artifacts, lifecycle mismatch, cost cap breach, theme boundary violation, canonical data write-path contact, or unexplained infrastructure errors.

In [24]:
required_fail_gates = [gate for gate in ('G2', 'G4', 'G5', 'G7', 'G8', 'G9') if GATES[gate]['status'] == 'FAIL']
any_failed = [gate for gate, info in GATES.items() if info['status'] == 'FAIL']
warn_gates = [gate for gate, info in GATES.items() if info['status'] == 'WARN']

if any_failed:
    verdict = 'FAIL'
    next_action = 'Patch failed smoke gate(s), rerun acceptance, and do not scale Phase 2C yet.'
elif warn_gates or artifact_path is None:
    verdict = 'PARTIAL PASS' if artifact_path is not None else 'PARTIAL / NOT_RUN'
    next_action = 'Provide or complete the smoke artifact set, then rerun this notebook before scaling.'
else:
    verdict = 'PASS'
    next_action = 'Proceed to the next controlled Phase 2C scale decision using this smoke evidence.'

mark_gate('G10', 'PASS', 'Final decision generated')

final_decision = {
    'verdict': verdict,
    'passed_gates': [gate for gate, info in GATES.items() if info['status'] in {'PASS', 'WARN'}],
    'failed_gates': [gate for gate, info in GATES.items() if info['status'] == 'FAIL'],
    'warnings': warnings,
    'inspected_artifact_path': inspected_artifact_path,
    'head_commit': head_commit,
    'next_recommended_action': next_action,
}
print(json.dumps(final_decision, indent=2, default=str))

{
  "verdict": "PARTIAL / NOT_RUN",
  "passed_gates": [
    "G1",
    "G2",
    "G3",
    "G4",
    "G5",
    "G6",
    "G7",
    "G8",
    "G9",
    "G10"
  ],
  "failed_gates": [],
  "warnings": [
    "Working tree has 5 dirty entries; review before treating this as immutable acceptance evidence.",
    "G3: No smoke artifact path configured or discovered",
    "G8: Artifact completeness not run because no artifact path was available",
    "G4: Lifecycle reconciliation not run; no summary loaded",
    "G5: No theme data available to validate rotation",
    "G6: Cost fields not available; cannot prove under cap",
    "G7: Failure review not run; no summary loaded"
  ],
  "inspected_artifact_path": null,
  "head_commit": "613da3f",
  "next_recommended_action": "Provide or complete the smoke artifact set, then rerun this notebook before scaling."
}
